In [266]:
import os
import enum

from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
from torch import Tensor
from torch import quasirandom as t_qrand

import botorch
from botorch import fit as b_fit
from botorch import models as b_models
from botorch import acquisition as b_acq
from botorch import optim as b_optim
from botorch import test_functions as b_tfuncs
from botorch.utils import transforms as b_transf
from botorch.posteriors import gpytorch as bp_gpytorch

import gpytorch
from gpytorch import likelihoods as g_lh
from gpytorch import constraints as g_constraints
from gpytorch import kernels as g_kern
from gpytorch import mlls as g_mlls

device: torch.device = torch.device("cpu" if torch.mps.is_available() else "cpu")
tensor_dtype: torch.dtype = torch.float32
runs: int = os.environ.get("SMOKE_TEST", 30)

dim: int = 3
num_samples: int = 5

lb: float = -5
ub: float = 5

confidence_level: float = 2

max_iterations: int = 30
max_cholesky_size: int = float("inf")

In [267]:
@dataclass
class BOState():
    max_iterations: int
    current_iteration: int = 1

    def is_exceeded(self):
        return self.current_iteration > self.max_iterations
    def increment(self):
        print(f"Current iteration: {self.current_iteration}")
        self.current_iteration = self.current_iteration + 1

In [268]:
ackley: b_tfuncs.Ackley = b_tfuncs.Ackley(
    dim=dim,
    negate=True,
    ).to(
        device=device, 
        dtype=tensor_dtype
        )
ackley.bounds[0, :].fill_(lb)
ackley.bounds[1, :].fill_(ub)

def eval_ackley(x: Tensor):
    return ackley(b_transf.unnormalize(x, bounds=ackley.bounds))

In [269]:
def create_surrogate(
    X: Tensor,
    Y: Tensor,
    noise_interval: Optional[g_constraints.Interval] = g_constraints.Interval(1e-8, 1e-3),
    matern_smoothness: float = 2.5,
) -> Tuple[b_models.SingleTaskGP, g_mlls.ExactMarginalLogLikelihood]:
    likelihood: g_lh.GaussianLikelihood = g_lh.GaussianLikelihood(
        noise_constraint=noise_interval
    )
    kernel: g_kern.ScaleKernel = g_kern.ScaleKernel(
        base_kernel=g_kern.MaternKernel(nu=matern_smoothness)
    )
    model: b_models.SingleTaskGP = b_models.SingleTaskGP(
        train_X=X,
        train_Y=Y,
        likelihood=likelihood,
        covar_module=kernel
    )
    mll: g_mlls.ExactMarginalLogLikelihood = g_mlls.ExactMarginalLogLikelihood(
        likelihood=likelihood,
        model=model
    )
    return (model, mll)

def get_uncertainty(
    model: b_models.SingleTaskGP,
    X: Tensor,
) -> Tensor:
    variances: Tensor = model.posterior(X=X).variance
    return torch.sqrt(variances) * confidence_level * 2

def sample_candidate_positions(
    dim: int,
    num_samples: int,
    scramble: Optional[bool] = True, 
) -> Tensor:
    sobol: t_qrand.SobolEngine = t_qrand.SobolEngine(
        dimension=dim,
        scramble=scramble,
        
    )
    return sobol.draw(n=num_samples).requires_grad_(True)

def get_ucb(
        model: b_models.SingleTaskGP, 
        X: Tensor,
        beta: float,
        ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean + beta * torch.sqrt(posterior.variance)
    
def get_lcb(
        model: b_models.SingleTaskGP, 
        X: Tensor,
        beta: float,
        ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean - beta * torch.sqrt(posterior.variance)

def get_minimizer_set_guess(
    model: b_models.SingleTaskGP,
    X: Tensor,
    Z: Tensor,
    W: Tensor,
) -> Tensor:
    lcb_X: Tensor = get_lcb(model=model, X=X, beta=confidence_level)
    ucb_Z: Tensor = get_ucb(model=model, X=Z, beta=confidence_level)

    safety: Tensor = torch.min(ucb_Z) - lcb_X
    masked: Tensor = safety >= 0

    minimized: Tensor = torch.where(masked, W, float("-inf"))
    return X[torch.argmax(minimized, dim=0)]

def get_expander_set_guess(
    model: b_models.SingleTaskGP,
    X: Tensor,
    Z: Tensor,
    W: Tensor,
) -> Tensor:
    eucl_distances: Tensor = torch.cdist(
        x1=X, 
        x2=Z
    )

    posterior_X: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    mean_flat: Tensor = posterior_X.mean.flatten()

    gradients: Tensor = torch.abs(torch.autograd.grad(
        outputs=mean_flat, 
        inputs=X, 
        grad_outputs=torch.ones_like(mean_flat), 
        retain_graph=True
    )[0])

    L_i: Tensor = torch.max(torch.abs(gradients)) # i-constraint Lipschitz constant
    ucb_i: Tensor = posterior_X.mean + confidence_level * torch.sqrt(posterior_X.variance)

    safety: Tensor = ucb_i - L_i * eucl_distances
    masked: Tensor = safety.any(dim=1, keepdim=True)

    maximized: Tensor = torch.where(masked, W, float("-inf"))
    return X[torch.argmax(maximized, dim=0)]

def select_next_candidate(
    model: b_models.SingleTaskGP,
    minimizer: Tensor,
    expander: Tensor,
) -> Tensor:
    cat: Tensor = torch.cat(
        (minimizer, expander), 
        dim=0
        )
    uncertainty: Tensor = get_uncertainty(
        model=model,
        X=cat
    )
    return cat[torch.argmax(uncertainty, dim=0)]

In [270]:
x_safe: Tensor = torch.rand(
    size=[num_samples, dim],
).to(device=device, dtype=tensor_dtype).requires_grad_(True)

y_safe: Tensor = torch.tensor(
    [eval_ackley(x_safe[i, :]) for i in range(num_samples)],
).to(device=device, dtype=tensor_dtype).unsqueeze(-1)

z_unsafe: Tensor = torch.zeros_like(
    x_safe,
    requires_grad=True
    ).to(device=device, dtype=tensor_dtype)

state: BOState = BOState(
    max_iterations=max_iterations,
)

while not state.is_exceeded():
    state.increment()
    model, mll = create_surrogate(
        X=x_safe.detach(),
        Y=y_safe.detach(),
    )
    
    with gpytorch.settings.max_cholesky_size(max_cholesky_size):
        b_fit.fit_gpytorch_mll(mll=mll)        
        x_candidates: Tensor = sample_candidate_positions(
            dim=dim,
            num_samples=num_samples,
            scramble=True,
        )
        z_candidates: Tensor = sample_candidate_positions(
            dim=dim,
            num_samples=num_samples,
            scramble=True,
        )

        uncertainty_candidates: Tensor = get_uncertainty(
            model=model,
            X=x_candidates,
        )

        minimizer: Tensor = get_minimizer_set_guess(
            model=model,
            X=x_candidates,
            Z=z_candidates,
            W=uncertainty_candidates
        )
        expander: Tensor = get_expander_set_guess(
            model=model,
            X=x_candidates,
            Z=z_candidates,
            W=uncertainty_candidates
        )

        next_candidates: Tensor = select_next_candidate(
            model=model,
            minimizer=minimizer,
            expander=expander,
        )

        y_next: Tensor = torch.tensor(
            [eval_ackley(x) for x in next_candidates], 
            dtype=tensor_dtype, 
            device=device
        ).unsqueeze(-1)

        x_safe: Tensor = torch.cat(
            (x_safe, next_candidates),
            dim=0,
        )
        y_safe: Tensor = torch.cat(
            (y_safe, y_next), 
            dim=0,
        )

        print(f"Uncertainty: {torch.amax(uncertainty_candidates):.4f}, " 
              f"minimum y value: {torch.amax(y_next)}")



Current iteration: 1
Uncertainty: 5.0227, minimum y value: -11.484381675720215
Current iteration: 2
Uncertainty: 4.8494, minimum y value: -12.297107696533203
Current iteration: 3
Uncertainty: 5.0643, minimum y value: -9.993178367614746
Current iteration: 4
Uncertainty: 4.8743, minimum y value: -9.648905754089355
Current iteration: 5


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use doubl

Uncertainty: 4.8008, minimum y value: -8.98278522491455
Current iteration: 6
Uncertainty: 4.9648, minimum y value: -3.96201229095459
Current iteration: 7
Uncertainty: 8.4187, minimum y value: -9.71213150024414
Current iteration: 8
Uncertainty: 8.0124, minimum y value: -12.714561462402344
Current iteration: 9
Uncertainty: 8.0398, minimum y value: -11.155230522155762
Current iteration: 10


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use doubl

Uncertainty: 6.9756, minimum y value: -10.238638877868652
Current iteration: 11
Uncertainty: 7.8150, minimum y value: -12.676151275634766
Current iteration: 12
Uncertainty: 4.6302, minimum y value: -11.083321571350098
Current iteration: 13


...

Uncertainty: 3.1807, minimum y value: -9.65338134765625
Current iteration: 24
Uncertainty: 2.8726, minimum y value: -5.167096138000488
Current iteration: 25
Uncertainty: 2.9399, minimum y value: -12.467118263244629
Current iteration: 26
Uncertainty: 2.2678, minimum y value: -10.3719482421875
Current iteration: 27


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use doubl

Uncertainty: 2.6435, minimum y value: -12.520069122314453
Current iteration: 28
Uncertainty: 2.2798, minimum y value: -8.795147895812988
Current iteration: 29
Uncertainty: 2.9815, minimum y value: -11.730691909790039
Current iteration: 30


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78859/3061935531.py:13: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use doubl

Uncertainty: 2.0787, minimum y value: -6.717438697814941


In [271]:
import plotly.graph_objects as go
import numpy as np
import torch

def plot_safeopt_objective_plotly(y_safe: torch.Tensor, num_initial_samples: int):
    # 1. Extract only the objective values evaluated DURING the loop
    # We flatten the array so Plotly can read it easily as a 1D list
    y_evaluated = y_safe[num_initial_samples:].detach().cpu().numpy().flatten()
    iterations = np.arange(1, len(y_evaluated) + 1)

    # 2. Initialize the Plotly figure
    fig = go.Figure()

    # 3. Add the SafeOpt objective line (expect high fluctuations!)
    fig.add_trace(
        go.Scatter(
            x=iterations, 
            y=y_evaluated, 
            mode='lines', 
            name='SafeOpt',
            line=dict(color='#1f77b4')
        )
    )

    # 4. Add the dashed red line for the 'minimum value' benchmark (0.0 for Ackley)
    fig.add_trace(
        go.Scatter(
            x=[iterations, iterations[-1]], 
            y=[0.0, 0.0], 
            mode='lines', 
            name='minimum value',
            line=dict(color='firebrick', dash='dash')
        )
    )

    # 5. Format the layout to match the paper's style
    fig.update_layout(
        title='SafeOpt Performance (Ackley Function)',
        xaxis_title='Iteration',
        yaxis_title='Objective',
        template='plotly_white', # Gives a clean background similar to matplotlib
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99
        )
    )
    
    fig.show(renderer="vscode") 

plot_safeopt_objective_plotly(
    y_safe=y_safe,
    num_initial_samples=num_samples
)